# ML-04 — Search Intelligence Data Contract

**Lane: Refresh / Content Opportunity Scoring.** This is a directional review queue, not an automated edit or a causal claim.

## 1. Contract in plain words

1. **One row:** one content item for one client on one report date in the daily fact table. My feature frame later collapses this to one row per client-content item.
2. **Table:** `fact_content_daily_performance` for daily Search Console measures; I deliberately do not join dimensions for this first pass.
3. **Time window:** March 1–31, 2026. Features use March 1–15; March 16–31 is a later within-month proxy window.
4. **Target/proxy:** `is_declining_proxy = 1` when second-half impressions are below 80% of first-half impressions, for items with at least 100 first-half impressions. This rule-defined proxy is not an observed post-refresh outcome.
5. **Deliberately excluded:** GA4 sessions/engagement from the features because GA4 availability varies; the third verification query measures that coverage rather than treating unavailable zero-filled values as real zeros.

**Human action:** an editor reviews the highest-priority eligible items first, then independently decides whether to refresh, expand, protect, or monitor. A false recommendation can waste review time; a missed item can delay useful attention.

In [3]:
import getpass
import os

import duckdb
import pandas as pd
from IPython.display import display

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN and "google.colab" in __import__("sys").modules:
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token (input is hidden): ")
if not HF_TOKEN:
    raise RuntimeError("A Hugging Face READ token is required to query the gated warehouse.")

con = duckdb.connect()
escaped_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{escaped_token}')")
del escaped_token, HF_TOKEN

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
print("Connected to the gated warehouse. Token is not displayed or saved in the notebook.")

Connected to the gated warehouse. Token is not displayed or saved in the notebook.


## 2. Fields: feature / label / context / excluded

- **Features (5):** first-half impressions, first-half clicks, first-half mean GSC position, first-half days with impressions, and first-half daily-impression standard deviation. Each is knowable at the March 15 decision moment because it uses March 1–15 observations only.
- **Label/proxy:** whether second-half impressions fall below 80% of first-half impressions, among items with at least 100 first-half impressions. It is only known after March 31 and is never a model input.
- **Context:** `report_date`, `client_hash_id`, and `content_hash_id` establish grain and support grouping/splitting. Hashes are never model inputs and are omitted from displayed row previews.
- **Excluded:** GA4 measures because coverage is incomplete; identifiers as predictive features; and all March 16–31 values from the feature set. No product-decision flags are used.

## 3. Three verification queries, then five features and the leakage trap

All three checks below read only the `month=2026-03` partition. Query 1 counts duplicate `(report_date, client_hash_id, content_hash_id)` groups; zero means the claimed daily grain holds. Query 2 proves the monthly row count and actual date bounds. Query 3 uses `ga4_data_available IS TRUE` and counts the surviving rows. Next, a separate feature query builds exactly five first-half signals. The deliberate leak column is an exact copy of the future-window proxy; its perfect score is shown, then that column is deleted before the honest score is retained.

In [4]:
grain_check = con.sql(f"""
    SELECT COUNT(*) AS duplicate_grain_groups
    FROM (
        SELECT report_date, client_hash_id, content_hash_id
        FROM {FACT_MARCH}
        GROUP BY report_date, client_hash_id, content_hash_id
        HAVING COUNT(*) > 1
    )
""").df()
print("Query 1 — duplicate daily grain groups (expected: 0):")
display(grain_check)

month_check = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS first_report_date,
           MAX(report_date) AS last_report_date
    FROM {FACT_MARCH}
""").df()
print("Query 2 — March row count and date span:")
display(month_check)

availability_check = con.sql(f"""
    SELECT (SELECT COUNT(*) FROM {FACT_MARCH}) AS march_rows,
           (SELECT COUNT(*) FROM {FACT_MARCH}
            WHERE ga4_data_available IS TRUE) AS rows_surviving_ga4_is_true
""").df()
print("Query 3 — rows available after the explicit GA4 IS TRUE check:")
display(availability_check)

Query 1 — duplicate daily grain groups (expected: 0):


,duplicate_grain_groups
0,0


Query 2 — March row count and date span:


,row_count,first_report_date,last_report_date
0,9841378,2026-03-01,2026-03-31


Query 3 — rows available after the explicit GA4 IS TRUE check:


,march_rows,rows_surviving_ga4_is_true
0,9841378,413966


**Feature availability, one line each:**

- `impressions_prev14`: knowable by March 15 because it sums GSC impressions from March 1–15.
- `clicks_prev14`: knowable by March 15 because it sums GSC clicks from March 1–15.
- `avg_position_prev14`: knowable by March 15 because it averages nonzero GSC positions from March 1–15 only.
- `active_search_days_prev14`: knowable by March 15 because it counts first-half dates with positive impressions.
- `impression_std_prev14`: knowable by March 15 because it measures first-half daily impression variability.

The target is computed separately from March 16–31; items need at least 100 first-half impressions. The leakage demonstration uses a client-group holdout so rows from a client cannot appear in both train and test.

In [6]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

features_df = con.sql(f"""
    WITH per_content AS (
        SELECT client_hash_id,
               content_hash_id,
               SUM(CASE WHEN report_date < DATE '2026-03-16'
                        THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_prev14,
               SUM(CASE WHEN report_date < DATE '2026-03-16'
                        THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clicks_prev14,
               AVG(CASE WHEN report_date < DATE '2026-03-16'
                         AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_prev14,
               COUNT(DISTINCT CASE WHEN report_date < DATE '2026-03-16'
                                    AND gsc_impressions > 0 THEN report_date END) AS active_search_days_prev14,
               STDDEV_POP(CASE WHEN report_date < DATE '2026-03-16'
                               THEN gsc_impressions END) AS impression_std_prev14,
               SUM(CASE WHEN report_date >= DATE '2026-03-16'
                        THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_outcome16
        FROM {FACT_MARCH}
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT *, (impressions_outcome16 < 0.8 * impressions_prev14)::INTEGER AS is_declining_proxy
    FROM per_content
    WHERE impressions_prev14 >= 100
""").df()

feature_columns = [
    "impressions_prev14", "clicks_prev14", "avg_position_prev14",
    "active_search_days_prev14", "impression_std_prev14",
]
features_df[feature_columns] = features_df[feature_columns].fillna(0)
print(f"Eligible content items with >=100 first-half impressions: {len(features_df):,}")
print(f"Decline-proxy prevalence in this eligible feature frame: {features_df['is_declining_proxy'].mean():.1%}")
print("One row = one client-content item. IDs are context only and omitted from this preview.")
display(features_df[feature_columns + ["is_declining_proxy"]].head())

X = features_df[feature_columns]
y = features_df["is_declining_proxy"]
groups = features_df["client_hash_id"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

leaky_frame = features_df[[*feature_columns, "is_declining_proxy"]].copy()
leaky_frame["deliberate_label_leak"] = leaky_frame["is_declining_proxy"]
leaky_model = DecisionTreeClassifier(random_state=42)
leaky_model.fit(leaky_frame.iloc[train_idx][[*feature_columns, "deliberate_label_leak"]], y.iloc[train_idx])
leaky_predictions = leaky_model.predict(leaky_frame.iloc[test_idx][[*feature_columns, "deliberate_label_leak"]])
leaky_accuracy = accuracy_score(y.iloc[test_idx], leaky_predictions)
print(f"Leaky held-out accuracy with deliberate target copy (invalid; expect 1.000): {leaky_accuracy:.3f}")

leaky_frame = leaky_frame.drop(columns="deliberate_label_leak")
honest_model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=25, random_state=42)
honest_model.fit(leaky_frame.iloc[train_idx][feature_columns], y.iloc[train_idx])
honest_predictions = honest_model.predict(leaky_frame.iloc[test_idx][feature_columns])
honest_accuracy = accuracy_score(y.iloc[test_idx], honest_predictions)
print(f"Honest client-held-out accuracy using only the five first-half features: {honest_accuracy:.3f}")
assert "deliberate_label_leak" not in leaky_frame.columns
print("Leakage trap removed. The retained result is the honest score above; accuracy is only a quick diagnostic, not proof of model quality.")

Eligible content items with >=100 first-half impressions: 77,540
Decline-proxy prevalence in this eligible feature frame: 28.5%
One row = one client-content item. IDs are context only and omitted from this preview.


,impressions_prev14,clicks_prev14,avg_position_prev14,active_search_days_prev14,impression_std_prev14,is_declining_proxy
0,233.0,1.0,4.398736,15,5.655283,0
1,3377.0,1.0,7.393698,15,56.673176,0
2,146.0,0.0,33.919595,15,7.995554,1
3,20580.0,102.0,3.774374,15,488.423587,0
4,993.0,6.0,4.655306,15,14.561135,0


Leaky held-out accuracy with deliberate target copy (invalid; expect 1.000): 1.000
Honest client-held-out accuracy using only the five first-half features: 0.833
Leakage trap removed. The retained result is the honest score above; accuracy is only a quick diagnostic, not proof of model quality.


## 4. Data limits

**Named limitation:** this is a short, single-month slice and a rule-defined half-month proxy. It cannot show that a refresh causes recovery, nor that this experiment generalizes across months. Client histories are unbalanced; GA4 is unavailable on some rows, so zero-filled values cannot be treated as observed zero engagement. March is used as a mid-panel development month; June 2026 remains the sealed final month.

## 5. Self-check

- [x] Contract states the unit, table, window, proxy, and deliberate exclusion.
- [x] Exactly three verification queries show outputs, including explicit `IS TRUE` availability filtering.
- [x] Five feature columns are shown with a decision-time rationale; hashes are context, not predictors.
- [x] The target-derived leak produces a score, is deleted, and the honest score is reported separately.
- [ ] Run top to bottom with approved warehouse access and inspect the outputs.
- [x] No token, raw client IDs, URLs, or private queries are included in notebook source or displayed outputs.
- [ ] Commit the executed notebook to your public repository, then submit its GitHub URL on the card.